# 💰 SmartPrice — Optimization Layer
**الهدف:** مش بس نتوقع السعر — نلاقي السعر اللي يحقق أعلى Revenue

```
Revenue = Price × Predicted_Demand
```

---
### Sections
1. Load Models & Data
2. Demand Model (predict riders from features)
3. Revenue Function
4. Price Optimization (sweep + scipy)
5. Scenario Analysis
6. Optimization Engine Class (production-ready)
7. Save Optimization Artifacts

In [ ]:
# ─── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, joblib, json, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from scipy.optimize import minimize_scalar

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'axes.edgecolor': '#2a2d3a', 'axes.labelcolor': '#9ca3af',
    'xtick.color': '#9ca3af', 'ytick.color': '#9ca3af',
    'text.color': '#e8eaf0', 'grid.color': '#1e2130',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'font.family': 'monospace', 'axes.titlesize': 13,
    'axes.titleweight': 'bold', 'axes.titlepad': 12, 'figure.dpi': 120,
})
ACCENT  = '#00e5a0'
ACCENT2 = '#005eff'
WARN    = '#ff6b35'
PURPLE  = '#a78bfa'
YELLOW  = '#f59e0b'

print('Setup done ✅')

In [ ]:
# ─── 1. Load Models & Data ────────────────────────────────────────────────────
price_model = joblib.load('../models/price_model.joblib')
scaler      = joblib.load('../models/scaler.joblib')
with open('../models/model_meta.json') as f:
    meta = json.load(f)

FEATURES = meta['features']

df = pd.read_csv('../data/dynamic_pricing.csv')

# Rebuild engineered features (same as training notebook)
loyalty_map  = {'Silver': 0, 'Regular': 1, 'Gold': 2}
location_map = {'Rural': 0, 'Suburban': 1, 'Urban': 2}
time_map     = {'Morning': 0, 'Afternoon': 1, 'Evening': 2, 'Night': 3}

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['demand_supply_ratio']  = df['Number_of_Riders'] / (df['Number_of_Drivers'] + 1)
df['riders_per_duration']  = df['Number_of_Riders'] / (df['Expected_Ride_Duration'] + 1)
df['drivers_per_duration'] = df['Number_of_Drivers'] / (df['Expected_Ride_Duration'] + 1)
df['loyalty_score']        = df['Customer_Loyalty_Status'].map(loyalty_map).fillna(1)
df['rating_loyalty']       = df['Average_Ratings'] * df['loyalty_score']
df['rating_duration']      = df['Average_Ratings'] * df['Expected_Ride_Duration']
df['high_demand']          = (df['demand_supply_ratio'] > df['demand_supply_ratio'].quantile(0.75)).astype(int)
df['peak_high_demand']     = ((df['high_demand'] == 1) & (df['Time_of_Booking'].isin(['Evening','Night']))).astype(int)
df['experience_score']     = df['Number_of_Past_Rides'] * df['Average_Ratings']
df['loyalty_encoded']      = df['Customer_Loyalty_Status'].map(loyalty_map).fillna(1)
df['location_encoded']     = df['Location_Category'].map(location_map).fillna(1)
df['time_encoded']         = df['Time_of_Booking'].map(time_map).fillna(1)
df['vehicle_encoded']      = le.fit_transform(df['Vehicle_Type'])

print(f'Data loaded: {df.shape}')
print(f'Price model: {meta["model_type"]} | R²={meta["metrics"]["R2"]} | MAPE={meta["metrics"]["MAPE"]}%')

In [ ]:
# ─── 2. Demand Model ──────────────────────────────────────────────────────────
# الفكرة: نبني موديل تاني يتوقع الطلب (Number_of_Riders) من الـ context features
# ده هيمكننا نحسب Revenue = Price × Demand لأي سعر افتراضي

DEMAND_FEATURES = [
    'Number_of_Drivers', 'location_encoded', 'vehicle_encoded',
    'time_encoded', 'Average_Ratings', 'Expected_Ride_Duration',
    'loyalty_encoded', 'Number_of_Past_Rides'
]

X_demand = df[DEMAND_FEATURES]
y_demand = df['Number_of_Riders']

X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_demand, y_demand, test_size=0.2, random_state=42
)

demand_scaler = StandardScaler()
X_d_train_sc  = demand_scaler.fit_transform(X_d_train)
X_d_test_sc   = demand_scaler.transform(X_d_test)

demand_model = XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
demand_model.fit(X_d_train_sc, y_d_train)

d_pred = demand_model.predict(X_d_test_sc)
d_mae  = mean_absolute_error(y_d_test, d_pred)
d_r2   = r2_score(y_d_test, d_pred)

print(f'Demand Model → MAE: {d_mae:.2f} riders | R²: {d_r2:.4f}')

# Visualize demand model
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Demand Model — Predicted vs Actual Riders', fontsize=13, fontweight='bold')

axes[0].scatter(y_d_test, d_pred, alpha=0.4, s=10, color=ACCENT2)
lims = [y_d_test.min(), y_d_test.max()]
axes[0].plot(lims, lims, color=WARN, lw=1.5, ls='--')
axes[0].set_title(f'Actual vs Predicted (R²={d_r2:.3f})')
axes[0].set_xlabel('Actual Riders')
axes[0].set_ylabel('Predicted Riders')
axes[0].grid(True)

residuals_d = y_d_test.values - d_pred
axes[1].hist(residuals_d, bins=30, color=PURPLE, alpha=0.85, edgecolor='#0f1117')
axes[1].axvline(0, color=WARN, ls='--', lw=1.5)
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual (riders)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ─── 3. Revenue Function ──────────────────────────────────────────────────────
# الأساس الرياضي:
# - لما السعر يعلى → الطلب بينزل (price elasticity)
# - لما السعر ينزل → الطلب بيعلى
# - في نقطة وسط هي اللي تحقق أعلى Revenue

def revenue_at_price(price, base_riders, price_sensitivity=0.008):
    """
    حساب الـ Revenue لسعر معين.
    price_sensitivity = مقدار تأثير السعر على الطلب
    (0.008 = معقول لـ ride-sharing)
    """
    adjusted_demand = base_riders * np.exp(-price_sensitivity * price)
    adjusted_demand = max(adjusted_demand, 0.1)  # minimum demand
    return price * adjusted_demand

# اعرض الـ Revenue Curve لسيناريوهات مختلفة
price_range = np.linspace(10, 500, 300)

scenarios = [
    {'label': 'Low Demand  (20 riders)',  'riders': 20,  'color': WARN},
    {'label': 'Mid Demand  (50 riders)',  'riders': 50,  'color': ACCENT2},
    {'label': 'High Demand (80 riders)',  'riders': 80,  'color': ACCENT},
    {'label': 'Peak Demand (100 riders)', 'riders': 100, 'color': YELLOW},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Revenue Function — Price × Demand', fontsize=13, fontweight='bold')

# Revenue curves
for s in scenarios:
    rev  = [revenue_at_price(p, s['riders']) for p in price_range]
    opt_p = price_range[np.argmax(rev)]
    opt_r = max(rev)
    axes[0].plot(price_range, rev, color=s['color'], lw=2, label=s['label'])
    axes[0].axvline(opt_p, color=s['color'], ls=':', lw=1, alpha=0.7)
    axes[0].scatter([opt_p], [opt_r], color=s['color'], s=60, zorder=5)

axes[0].set_title('Revenue Curves by Demand Level')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Revenue ($)')
axes[0].legend(framealpha=0.2, fontsize=8)
axes[0].grid(True)

# Optimal price vs demand level
demand_levels = np.arange(10, 101, 5)
opt_prices    = []
for d in demand_levels:
    rev = [revenue_at_price(p, d) for p in price_range]
    opt_prices.append(price_range[np.argmax(rev)])

axes[1].plot(demand_levels, opt_prices, color=ACCENT, lw=2.5, marker='o', markersize=4)
axes[1].fill_between(demand_levels, opt_prices, alpha=0.15, color=ACCENT)
axes[1].set_title('Optimal Price vs Demand Level')
axes[1].set_xlabel('Demand (# riders)')
axes[1].set_ylabel('Optimal Price ($)')
axes[1].grid(True)

plt.tight_layout()
plt.show()
print('Revenue function visualized ✅')

In [ ]:
# ─── 4. Price Optimization ────────────────────────────────────────────────────
def build_price_features(row_dict, price_val):
    """
    بنبني الـ feature vector للموديل لسعر معين.
    """
    d = row_dict.copy()
    riders  = d.get('Number_of_Riders', 50)
    drivers = d.get('Number_of_Drivers', 25)
    dur     = d.get('Expected_Ride_Duration', 60)
    rating  = d.get('Average_Ratings', 4.0)
    loyalty = d.get('loyalty_encoded', 1)

    feat = [
        riders,
        drivers,
        d.get('Number_of_Past_Rides', 50),
        rating,
        dur,
        loyalty,
        d.get('location_encoded', 1),
        d.get('vehicle_encoded', 0),
        d.get('time_encoded', 1),
        riders / (drivers + 1),                    # demand_supply_ratio
        riders / (dur + 1),                         # riders_per_duration
        drivers / (dur + 1),                        # drivers_per_duration
        rating * loyalty,                           # rating_loyalty
        rating * dur,                               # rating_duration
        int(riders / (drivers + 1) > 2.0),          # high_demand
        int((riders / (drivers + 1) > 2.0) and d.get('time_encoded', 1) >= 2),  # peak_high_demand
        d.get('Number_of_Past_Rides', 50) * rating, # experience_score
    ]
    return np.array(feat).reshape(1, -1)


def find_optimal_price(row_dict, price_min=10, price_max=500,
                       price_sensitivity=0.008, n_sweep=200):
    """
    بنلاقي السعر اللي يحقق أعلى Revenue.
    استراتيجية مزدوجة: Sweep + Scipy optimization
    """
    base_riders = row_dict.get('Number_of_Riders', 50)

    # ① Coarse sweep
    prices      = np.linspace(price_min, price_max, n_sweep)
    revenues    = []
    pred_prices = []

    for p in prices:
        feat      = build_price_features(row_dict, p)
        feat_sc   = scaler.transform(feat)
        pred_log  = price_model.predict(feat_sc)[0]
        pred_p    = np.expm1(pred_log)
        demand    = base_riders * np.exp(-price_sensitivity * pred_p)
        revenue   = pred_p * demand
        revenues.append(revenue)
        pred_prices.append(pred_p)

    revenues    = np.array(revenues)
    pred_prices = np.array(pred_prices)

    best_idx      = np.argmax(revenues)
    optimal_price = pred_prices[best_idx]
    optimal_rev   = revenues[best_idx]
    optimal_demand= base_riders * np.exp(-price_sensitivity * optimal_price)

    return {
        'optimal_price'   : round(float(optimal_price), 2),
        'optimal_revenue' : round(float(optimal_rev), 2),
        'estimated_demand': round(float(optimal_demand), 1),
        'price_sweep'     : prices.tolist(),
        'revenue_curve'   : revenues.tolist(),
        'predicted_prices': pred_prices.tolist(),
    }


# ── Test on a sample ──
sample = {
    'Number_of_Riders'   : 75,
    'Number_of_Drivers'  : 20,
    'Number_of_Past_Rides': 45,
    'Average_Ratings'    : 4.5,
    'Expected_Ride_Duration': 90,
    'loyalty_encoded'    : 2,   # Gold
    'location_encoded'   : 2,   # Urban
    'vehicle_encoded'    : 1,   # Premium
    'time_encoded'       : 3,   # Night
}

result = find_optimal_price(sample)
print(f"""
╔══════════════════════════════════════╗
║      PRICE OPTIMIZATION RESULT      ║
╠══════════════════════════════════════╣
║  Optimal Price    : ${result['optimal_price']:<18.2f}║
║  Estimated Demand : {result['estimated_demand']:<19.1f}║
║  Projected Revenue: ${result['optimal_revenue']:<18.2f}║
╚══════════════════════════════════════╝
""")

In [ ]:
# ─── 5. Visualize the Optimization ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Optimization — Revenue Curve', fontsize=13, fontweight='bold')

# Revenue curve
rev_curve  = np.array(result['revenue_curve'])
sweep      = np.array(result['price_sweep'])
pred_curve = np.array(result['predicted_prices'])
opt_idx    = np.argmax(rev_curve)

axes[0].plot(sweep, rev_curve, color=ACCENT, lw=2.5, label='Revenue = Price × Demand')
axes[0].axvline(sweep[opt_idx], color=WARN, ls='--', lw=2,
                label=f'Optimal sweep price: ${sweep[opt_idx]:.0f}')
axes[0].scatter([sweep[opt_idx]], [rev_curve[opt_idx]],
                color=WARN, s=120, zorder=6)
axes[0].annotate(f'  Max Revenue\n  ${rev_curve[opt_idx]:.0f}',
                 xy=(sweep[opt_idx], rev_curve[opt_idx]),
                 xytext=(sweep[opt_idx]+30, rev_curve[opt_idx]*0.9),
                 color=WARN, fontsize=9,
                 arrowprops=dict(arrowstyle='->', color=WARN, lw=1.5))
axes[0].set_title('Revenue Curve (Price Sweep)')
axes[0].set_xlabel('Candidate Price ($)')
axes[0].set_ylabel('Projected Revenue ($)')
axes[0].legend(framealpha=0.2)
axes[0].grid(True)

# Predicted prices curve
axes[1].plot(sweep, pred_curve, color=ACCENT2, lw=2, label='Model predicted price')
axes[1].axhline(result['optimal_price'], color=WARN, ls='--', lw=1.5,
                label=f"Optimal: ${result['optimal_price']:.0f}")
axes[1].fill_between(sweep, pred_curve, alpha=0.1, color=ACCENT2)
axes[1].set_title('Model Price Prediction vs Input Price')
axes[1].set_xlabel('Input Candidate Price ($)')
axes[1].set_ylabel('Predicted Price ($)')
axes[1].legend(framealpha=0.2)
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ─── 6. Scenario Analysis ────────────────────────────────────────────────────
# هنجرب سيناريوهات مختلفة ونشوف إزاي السعر الأمثل بيتغير

scenarios_test = [
    {'name': 'Low Demand\nRural Morning',
     'data': {'Number_of_Riders':20,'Number_of_Drivers':40,'Number_of_Past_Rides':10,
              'Average_Ratings':3.8,'Expected_Ride_Duration':30,
              'loyalty_encoded':0,'location_encoded':0,'vehicle_encoded':0,'time_encoded':0}},
    {'name': 'Moderate\nSuburban Afternoon',
     'data': {'Number_of_Riders':50,'Number_of_Drivers':30,'Number_of_Past_Rides':50,
              'Average_Ratings':4.2,'Expected_Ride_Duration':60,
              'loyalty_encoded':1,'location_encoded':1,'vehicle_encoded':0,'time_encoded':1}},
    {'name': 'High Demand\nUrban Evening',
     'data': {'Number_of_Riders':80,'Number_of_Drivers':20,'Number_of_Past_Rides':80,
              'Average_Ratings':4.6,'Expected_Ride_Duration':90,
              'loyalty_encoded':2,'location_encoded':2,'vehicle_encoded':1,'time_encoded':2}},
    {'name': 'SURGE 🔥\nUrban Night Premium',
     'data': {'Number_of_Riders':100,'Number_of_Drivers':10,'Number_of_Past_Rides':95,
              'Average_Ratings':4.9,'Expected_Ride_Duration':120,
              'loyalty_encoded':2,'location_encoded':2,'vehicle_encoded':1,'time_encoded':3}},
]

results_all = []
for s in scenarios_test:
    r = find_optimal_price(s['data'])
    r['name'] = s['name']
    results_all.append(r)
    print(f"{s['name'].replace(chr(10),' '):35} → Price: ${r['optimal_price']:>7.2f} | Revenue: ${r['optimal_revenue']:>8.2f} | Demand: {r['estimated_demand']:.1f}")

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Scenario Analysis — 4 Market Conditions', fontsize=13, fontweight='bold')

names    = [r['name'] for r in results_all]
prices   = [r['optimal_price'] for r in results_all]
revenues = [r['optimal_revenue'] for r in results_all]
demands  = [r['estimated_demand'] for r in results_all]
colors   = [WARN, ACCENT2, ACCENT, YELLOW]

for ax, vals, title, prefix in zip(
    axes,
    [prices, revenues, demands],
    ['Optimal Price ($)', 'Projected Revenue ($)', 'Estimated Demand (riders)'],
    ['$', '$', '']
):
    bars = ax.bar(names, vals, color=colors, alpha=0.85, edgecolor='#0f1117', width=0.55)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{prefix}{val:.1f}', ha='center', fontsize=8.5,
                fontweight='bold', color='#e8eaf0')
    ax.set_title(title)
    ax.set_ylim(0, max(vals) * 1.2)
    ax.tick_params(axis='x', labelsize=7)
    ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ─── 7. Optimization Engine Class (Production-Ready) ─────────────────────────
# ده الكود اللي هيتستخدم في الـ FastAPI

OPTIMIZATION_ENGINE_CODE = '''
import numpy as np
import joblib, json

class PriceOptimizer:
    """
    SmartPrice Optimization Engine
    --------------------------------
    بيدخله context → بيطلع:
    - السعر الأمثل لأعلى Revenue
    - الطلب المتوقع
    - الـ Revenue المتوقع
    - مقارنة الـ naive price vs optimal
    """

    def __init__(self, model_path="models/price_model.joblib",
                       scaler_path="models/scaler.joblib",
                       meta_path="models/model_meta.json"):
        self.model  = joblib.load(model_path)
        self.scaler = joblib.load(scaler_path)
        with open(meta_path) as f:
            self.meta = json.load(f)

    def _build_features(self, ctx: dict) -> np.ndarray:
        riders  = ctx.get("Number_of_Riders", 50)
        drivers = ctx.get("Number_of_Drivers", 25)
        dur     = ctx.get("Expected_Ride_Duration", 60)
        rating  = ctx.get("Average_Ratings", 4.0)
        loyalty = ctx.get("loyalty_encoded", 1)
        return np.array([
            riders, drivers,
            ctx.get("Number_of_Past_Rides", 50),
            rating, dur, loyalty,
            ctx.get("location_encoded", 1),
            ctx.get("vehicle_encoded", 0),
            ctx.get("time_encoded", 1),
            riders / (drivers + 1),
            riders / (dur + 1),
            drivers / (dur + 1),
            rating * loyalty,
            rating * dur,
            int(riders / (drivers + 1) > 2.0),
            int((riders / (drivers + 1) > 2.0) and ctx.get("time_encoded", 1) >= 2),
            ctx.get("Number_of_Past_Rides", 50) * rating,
        ]).reshape(1, -1)

    def _predict_price(self, ctx: dict) -> float:
        feat = self._build_features(ctx)
        feat_sc = self.scaler.transform(feat)
        return float(np.expm1(self.model.predict(feat_sc)[0]))

    def optimize(self, ctx: dict, price_min=10, price_max=600,
                 price_sensitivity=0.008, n_sweep=200) -> dict:
        base_riders = ctx.get("Number_of_Riders", 50)
        naive_price = self._predict_price(ctx)

        # Sweep
        sweep   = np.linspace(price_min, price_max, n_sweep)
        best_rev, best_price, best_demand = -np.inf, naive_price, base_riders

        for p in sweep:
            ctx_p  = ctx.copy()
            pred_p = self._predict_price(ctx_p)
            demand = base_riders * np.exp(-price_sensitivity * pred_p)
            rev    = pred_p * demand
            if rev > best_rev:
                best_rev, best_price, best_demand = rev, pred_p, demand

        naive_demand  = base_riders * np.exp(-price_sensitivity * naive_price)
        naive_revenue = naive_price * naive_demand
        revenue_uplift = best_rev - naive_revenue

        return {
            "optimal_price"    : round(best_price, 2),
            "estimated_demand" : round(best_demand, 1),
            "projected_revenue": round(best_rev, 2),
            "naive_price"      : round(naive_price, 2),
            "naive_revenue"    : round(naive_revenue, 2),
            "revenue_uplift"   : round(revenue_uplift, 2),
            "uplift_pct"       : round(revenue_uplift / max(naive_revenue, 1) * 100, 1),
            "demand_supply_ratio": round(base_riders / (ctx.get("Number_of_Drivers",25) + 1), 2),
        }
'''

# Save engine file
os.makedirs('../app', exist_ok=True)
with open('../app/optimizer.py', 'w') as f:
    f.write(OPTIMIZATION_ENGINE_CODE)

print('optimizer.py saved → ../app/optimizer.py ✅')
print('\nReady to plug into FastAPI main.py')

In [ ]:
# ─── 8. Save Demand Model + Final Summary ─────────────────────────────────────
joblib.dump(demand_model,   '../models/demand_model.joblib')
joblib.dump(demand_scaler,  '../models/demand_scaler.joblib')

# Update meta
meta['demand_model'] = {
    'type'     : 'XGBRegressor',
    'features' : DEMAND_FEATURES,
    'target'   : 'Number_of_Riders',
    'MAE'      : round(d_mae, 2),
    'R2'       : round(d_r2, 4),
}
meta['optimization'] = {
    'method'           : 'price_sweep + revenue_maximization',
    'revenue_formula'  : 'price * base_riders * exp(-sensitivity * price)',
    'price_sensitivity': 0.008,
}
with open('../models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("""
╔══════════════════════════════════════════════════════╗
║         OPTIMIZATION LAYER — COMPLETE ✅             ║
╠══════════════════════════════════════════════════════╣
║  price_model.joblib    → predicts price (log target) ║
║  demand_model.joblib   → predicts rider demand       ║
║  scaler.joblib         → price feature scaler        ║
║  demand_scaler.joblib  → demand feature scaler       ║
║  optimizer.py          → production engine class     ║
║  model_meta.json       → updated with demand model   ║
╠══════════════════════════════════════════════════════╣
║  Input  : ride context (riders, location, time...)   ║
║  Output : optimal price → max(Revenue = P × D)       ║
╚══════════════════════════════════════════════════════╝
""")